In [1]:
import numpy as np
import tkinter as tk
from PIL import Image, ImageTk
import os
from typing import List, Optional


class SimpleThompsonSampling:
    """Класс для реализации алгоритма сэмплирования Томпсона для рекомендации изображений."""

    def __init__(self, num_images: int) -> None:
        """Инициализирует счетчики лайков и дизлайков для каждого изображения."""
        self.num_images = num_images
        self.alpha = np.ones(
            num_images
        )  # счётчик лайков (α параметры бета-распределения)
        self.beta = np.ones(
            num_images
        )  # счётчик дизлайков (β параметры бета-распределения)

    def recommend_image(self) -> int:
        """Рекомендует изображение на основе текущего состояния алгоритма."""
        sampled_probs = np.random.beta(self.alpha, self.beta)
        return np.argmax(sampled_probs)

    def update_feedback(self, image_idx: int, liked: bool) -> None:
        """Обновляет статистику по изображению на основе пользовательского фидбека."""
        if liked:
            self.alpha[image_idx] += 1
        else:
            self.beta[image_idx] += 1


class ImageRecommenderApp:
    """Графический интерфейс для системы рекомендации изображений."""

    def __init__(self, root: tk.Tk, image_folder: str) -> None:
        """Инициализирует интерфейс и загружает изображения."""
        self.root = root
        self.root.title("Рекомендация изображений")

        # Загрузка изображений из папки
        self.image_files = [
            f
            for f in os.listdir(image_folder)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]
        self.num_images = len(self.image_files)

        if self.num_images == 0:
            raise ValueError("В папке нет изображений!")

        # Инициализация Thompson Sampling
        self.recommender = SimpleThompsonSampling(self.num_images)

        # Загрузка и ресайз изображений
        self.images: List[ImageTk.PhotoImage] = []
        for img_file in self.image_files:
            img_path = os.path.join(image_folder, img_file)
            img = Image.open(img_path)
            img = img.resize((300, 300), Image.LANCZOS)
            self.images.append(ImageTk.PhotoImage(img))

        # GUI элементы
        self.canvas = tk.Canvas(root, width=400, height=400)
        self.canvas.pack()

        self.like_btn = tk.Button(
            root,
            text="Лайк 👍",
            command=self.like,
            bg="green",
            fg="white",
            font=("Arial", 14),
        )
        self.like_btn.pack(side=tk.LEFT, padx=20, pady=10)

        self.dislike_btn = tk.Button(
            root,
            text="Дизлайк 👎",
            command=self.dislike,
            bg="red",
            fg="white",
            font=("Arial", 14),
        )
        self.dislike_btn.pack(side=tk.RIGHT, padx=20, pady=10)

        # Статистика
        self.stats_label = tk.Label(root, text="", font=("Arial", 12))
        self.stats_label.pack()

        self.current_image_idx: Optional[int] = None
        self.show_next_image()

    def show_next_image(self) -> None:
        """Показывает следующее рекомендованное изображение и обновляет статистику."""
        self.current_image_idx = self.recommender.recommend_image()

        # Очищаем canvas и показываем изображение
        self.canvas.delete("all")
        self.canvas.create_image(
            200, 200, image=self.images[self.current_image_idx]
        )  # Центрируем

        # Обновляем статистику
        self.update_stats()

    def like(self) -> None:
        """Обрабатывает действие "лайк" и показывает следующее изображение."""
        self.recommender.update_feedback(self.current_image_idx, liked=True)
        self.show_next_image()

    def dislike(self) -> None:
        """Обрабатывает действие "дизлайк" и показывает следующее изображение."""
        self.recommender.update_feedback(self.current_image_idx, liked=False)
        self.show_next_image()

    def update_stats(self) -> None:
        """Обновляет отображаемую статистику по всем изображениям."""
        stats_text = "Статистика:\n"
        for i in range(self.num_images):
            likes = int(self.recommender.alpha[i] - 1)
            dislikes = int(self.recommender.beta[i] - 1)
            stats_text += f"{self.image_files[i]}: 👍{likes} 👎{dislikes}\n"
        self.stats_label.config(text=stats_text)

In [2]:
root = tk.Tk()
app = ImageRecommenderApp(root, image_folder="origins/task_2")
root.mainloop()